In [ ]:
import pandas as pd
import os
import tensorflow as tf
from tensorflow.keras.layers import Dense, Input, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Model
from transformers import TFBertModel
import transformers
import numpy as np

In [ ]:
df = pd.read_csv("../input/feedback-prize-effectiveness/train.csv")

In [ ]:
AUTO = tf.data.experimental.AUTOTUNE
# Configuration
EPOCHS = 3
BATCH_SIZE = 8
MAX_LEN = 128

In [ ]:
eff = pd.read_csv('../input/compet00/ef.csv')
ineff = pd.read_csv('../input/compet00/inef.csv')
adq = pd.read_csv('../input/compet00/ad (1).csv')

In [ ]:
adq2 = []
for i in adq['0']:
    adq2.append(i)

In [ ]:
eff2 = []
for i in eff['0']:
    eff2.append(i)

In [ ]:
ineff2 = []
for i in ineff['0']:
    ineff2.append(i)

In [ ]:
all = adq2 + eff2 + ineff2

In [ ]:
tokens = [sentence.split() for sentence in df['discourse_text'][df['discourse_effectiveness'] == 'Adequate']]
Newad = []
for token in tokens:
    included_words = ''.join(str(word.lower()+' ') if word.lower() in all else '' for word in token)
    Newad.append(included_words)

In [ ]:
df['discourse_text'][df['discourse_effectiveness'] == 'Adequate'] = Newad

In [ ]:
tokens = [sentence.split() for sentence in df['discourse_text'][df['discourse_effectiveness'] == 'Effective']]
Newef = []
for token in tokens:
    included_words = ''.join(str(word.lower()+' ') if word.lower() in all else '' for word in token)
    Newef.append(included_words)

In [ ]:
df['discourse_text'][df['discourse_effectiveness'] == 'Effective'] = Newef

In [ ]:
tokens = [sentence.split() for sentence in df['discourse_text'][df['discourse_effectiveness'] == 'Ineffective']]
Newinef = []
for token in tokens:
    included_words = ''.join(str(word.lower()+' ') if word.lower() in all else '' for word in token)
    Newinef.append(included_words)

In [ ]:
df['discourse_text'][df['discourse_effectiveness'] == 'Ineffective'] = Newinef

In [ ]:
s0 = []
for i in df['discourse_text']:
    if len(i.split()) > 2:
        s0.append(i)
    else:
       s0.append('0')

In [ ]:
df['discourse_text'] = s0

In [ ]:
df = df[df.discourse_text != '0']

In [ ]:
def bert_encode(texts, tokenizer, max_len=MAX_LEN):
    input_ids = []
    token_type_ids = []
    attention_mask = []
    
    for text in texts:
        token = tokenizer(text, max_length=max_len, truncation=True, padding='max_length',
                         add_special_tokens=True)
        input_ids.append(token['input_ids'])
        token_type_ids.append(token['token_type_ids'])
        attention_mask.append(token['attention_mask'])
    
    return np.array(input_ids), np.array(token_type_ids), np.array(attention_mask)

In [ ]:
# First load the real tokenizer
tokenizer = transformers.BertTokenizer.from_pretrained('../input/huggingface-bert-variants/bert-base-cased/bert-base-cased')
# Save the loaded tokenizer locally
tokenizer.save_pretrained('.')

In [ ]:
new_label = {"discourse_effectiveness": {"Ineffective": 0, "Adequate": 1, "Effective": 2}}
df = df.replace(new_label)
df = df.rename(columns = {"discourse_effectiveness": "label"})

In [ ]:
sep = tokenizer.sep_token

In [ ]:
df['inputs'] = df.discourse_type + sep + df.discourse_text

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_valid, y_train, y_valid = train_test_split(df['inputs'], df['label'], test_size=0.01, random_state=42)

In [ ]:
X_train = bert_encode(X_train.astype(str), tokenizer)
X_valid = bert_encode(X_valid.astype(str), tokenizer)

y_train = y_train.values
y_valid = y_valid.values

In [ ]:
train_dataset = (
    tf.data.Dataset
    .from_tensor_slices((X_train, y_train))
    .repeat()
    .shuffle(2048)
    .batch(BATCH_SIZE)
    .prefetch(AUTO)
)

valid_dataset = (
    tf.data.Dataset
    .from_tensor_slices((X_valid, y_valid))
    .batch(BATCH_SIZE)
    .cache()
    .prefetch(AUTO)
)

In [ ]:
def build_model(bert_model, max_len=MAX_LEN):    
    input_ids = Input(shape=(max_len,), dtype=tf.int32, name="input_ids")
    token_type_ids = Input(shape=(max_len,), dtype=tf.int32, name="token_type_ids")
    attention_mask = Input(shape=(max_len,), dtype=tf.int32, name="attention_mask")

    sequence_output = bert_model(input_ids, token_type_ids=token_type_ids, attention_mask=attention_mask)[0]
    clf_output = sequence_output[:, 0, :]
    clf_output = Dropout(.1)(clf_output)
    out = Dense(3, activation='softmax')(clf_output)
    
    model = Model(inputs=[input_ids, token_type_ids, attention_mask], outputs=out)
    model.compile(Adam(lr=1e-5), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    
    return model

In [ ]:
%%time
transformer_layer = (TFBertModel.from_pretrained('../input/huggingface-bert-variants/bert-base-cased/bert-base-cased'))
model = build_model(transformer_layer, max_len=MAX_LEN)

In [ ]:
train_history = model.fit(
    train_dataset,
    steps_per_epoch=150,
    validation_data=valid_dataset,
    epochs=5
)

In [ ]:
test = pd.read_csv("../input/feedback-prize-effectiveness/test.csv")

In [ ]:
test['text'] = df.discourse_type + sep + df.discourse_text
test.head()

In [ ]:
test_text = bert_encode(test.text.astype(str), tokenizer)

In [ ]:
preds = model.predict(test_text, verbose=1)

In [ ]:
sample = pd.read_csv("../input/feedback-prize-effectiveness/sample_submission.csv")
sample.head()

In [ ]:
sample['Adequate'] = preds[:, 0]
sample['Effective'] = preds[:, 1]
sample['Ineffective'] = preds[:, 2]

sample.head(10)

In [ ]:
#sample.to_csv('submission.csv', index=False)